# Trial of Reflection


### Part 1 - Elijah

In [ ]:
import numpy as np
# 4 sensings, 3 binary tells each
tells = np.array([[1, 0, 1], # foot shift, no guard drop, exhale
    [0, 1, 1], # no shift, guard drop, exhale
    [0, 0, 1], # only exhale
    [1, 1, 1]]) # all three (the bluff)
# Ground truth: 1 = strike imminent, 0 = they will hold
strike = np.array([[1, 1, 0, 0]]).T # column vector, shape (4, 1)

SEED = 42



# Train single layer network
def single_layer_train(tells, strike, alpha, epochs, seed):
    # set seed for reproducibility
    np.random.seed(seed)

    # init a vector of weights of size (3,)
    weights = 3 * np.random.random(3)

    # for every epoch
    for i in range(epochs):
        # for every row in the tells vector
        for j in range(len(tells)):
            # get pred for this round
            pred = tells[j] @ weights

            # calculate the delta
            delta = pred - strike[j]

            # calculate weight deltas (delta * input)
            weight_deltas = delta * tells[j]

            # new weights = old weights - alpha * weight_deltas
            weights -= alpha * weight_deltas

    print(" Foot shift, guard drop, exhale ")
    print(weights)

    # init total error
    total_error = 0

    # Printing final four preds vs goals; also calculate errors
    for i in range(len(tells)):
        pred = tells[i] @ weights.T
        print(f"Pred = {pred}, goal was {strike[i]}")

        #error = delta ^ 2
        error = (pred - strike[i]) ** 2
        total_error += error

    return total_error


        


    
    


"""
This fails because, for this data, there is not a single input that is telling the story on its own.
Looking at the columns in tells every input has an equivalent number of strikes to holds when present
and when not present (i.e. foot shift when present had a strike one time and hold the other, and the same
thing when not present; this goes for guard drop, too, and exhale is just always present so it
means nothing valuable.) The valuable relationship is the XOR relationship between foot strike and guard
drop which is NOT representable through a simple linear weighted sum.
"""




if __name__ == '__main__':
    single_layer_train(tells, strike, .1, 60, SEED)

### Part 2 - Robert

In [ ]:
import numpy as np
np.random.seed(1)
def relu(x): return (x > 0) * x # max(0, x) elementwise

### Initial Dataset ###
# 4 sensings, 3 binary tells each
tells = np.array([[1, 0, 1], # foot shift, no guard drop, exhale
                    [0, 1, 1], # no shift, guard drop, exhale
                    [0, 0, 1], # only exhale
                    [1, 1, 1]]) # all three (the bluff)
# Ground truth: 1 = strike imminent, 0 = they will hold
strike = np.array([[1, 1, 0, 0]]).T # column vector, shape (4, 1)
# it does not appear that strike is used in this code



def forward(layer_0, weights_0_1, weights_1_2):
    # First weighted sum, followed by ReLU
    layer_1 = relu(layer_0.dot(weights_0_1)) # shape (1, 4)
    # Second weighted sum, no output activation yet
    layer_2 = layer_1.dot(weights_1_2) # shape (1, 1)
    return (layer_1, layer_2)

# Initializing our weights
hidden_size = 4
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1 # 3 * 4
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1 # 4 * 1

# Pass all 4 sensings through forward() and print the results
for i, sensing in enumerate(tells): # No longer initializing layer 0. 
    # sensing in enumerate(tells) allows for the properinitialization 
    # of layer 0 without ignoring layers 3 and 4. Previous attempts 
    # where I defined layer 0 would lead to the program ignoring layers 3 and 4
    
    layer_1, layer_2 = forward(sensing, weights_0_1, weights_1_2)

    print(f"Sensing {i + 1}:") # Prints and Sensing 1, Sensing 2, etc.
    # Print each layer of the Sensing
    print("Layer 1:", layer_1)
    print("Layer 2:", layer_2)
    # Empty space to organize by Sensing
    print()

## Comment Block

"""
Layer 0 is a (1, 3) matrix that shows the row it is currently viewing in tells.
Layer 1 is a (1, 4) matrix that holds 4 spearate values as a result of multiplying 
the first row in tells by the 3*4 matrix from the weights. The layer 2 matrix is 
a 1*1 matrix resulting from multiplying layer 1 with the 4*1 matrix from the weights.

To plug things in on how we get from layer 0 to layer 2 with the weight matrices, we would start with 
(1, 3) * (3, 4). We can cancel out the middle 3s and we end up with a (1, 4) matrix which is what layer 
1 is the (3, 4) matrix is the weights_0_1 in this equation. The reason we canel out the middle numbers is 
because they must match to perform dot multiplication and the resulting Matrix is always the outer values. 
From this, we do (1, 4) * (4, 1). (4, 1) is the other weight matrix. Canceling out the 4s, we end up with 
a (1, 1) matrix which is what layer 2 is.
"""

### Part 3 - Colson

In [ ]:
import numpy as np

np.random.seed(1)


def relu(x):
    return (x > 0) * x


def relu2deriv(y):
    return (y > 0).astype(int)


def one_step(layer_0, target, weights_0_1, weights_1_2, alpha):
    # Forward propogate
    layer_1 = relu(layer_0.dot(weights_0_1))
    layer_2 = layer_1.dot(weights_1_2)

    # Calculate error
    layer_2_error = np.sum((layer_2 - target) ** 2)

    # Back propogate
    layer_2_delta = layer_2 - target
    layer_1_delta = (
        layer_2_delta.dot(weights_1_2.T)
        * relu2deriv(layer_1)
    )

    # Update weights
    weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
    weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

    # return weights
    return weights_0_1, weights_1_2, layer_2, layer_2_error


tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
])

strike = np.array([[1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 4

weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1


layer_0 = tells[0:1]
target = strike[0:1]


layer_1_before = relu(layer_0.dot(weights_0_1))
layer_2_before = layer_1_before.dot(weights_1_2)
layer_2_error_before = np.sum((layer_2_before - target) ** 2)

print("Layer 2 before:", layer_2_before)
print("Layer 2 error before:", layer_2_error_before)
print("Weights 0-1 shape before:", weights_0_1.shape)
print("Weights 1-2 shape before:", weights_1_2.shape)


weights_0_1, weights_1_2, layer_2, layer_2_error = one_step(layer_0, target, weights_0_1, weights_1_2, alpha)


layer_1_after = relu(layer_0.dot(weights_0_1))
layer_2_after = layer_1_after.dot(weights_1_2)
layer_2_error_after = np.sum((layer_2_after - target) ** 2)

print("Layer 2 after:", layer_2_after)
print("Layer 2 error after:", layer_2_error_after)
print("Weights 0-1 shape after:", weights_0_1.shape)
print("Weights 1-2 shape after:", weights_1_2.shape)

# ------------------------------------------------- PART 4 -----------------------------------------------------
# Manual verification for weights_1_2[1, 0]:
# new = old - alpha * layer_1.T.dot(layer_2_delta)
# new = 0.75623487 - 0.2 * (0.51828245 * -0.60805673)
# new = 0.81926390

# ------------------------------------------------- PART 5 -----------------------------------------------------
# We transpose the weights because we're propagating the error backwards so the transpose reverses the order 
# so that the output error can distribute the error to the hidden neurons. We multiply by relu2deriv(layer 1)
# to only use the neurons that were active after relu. Anything that equals 0 doesn't affect anything.

### Part 4 - Kevin

In [ ]:
import numpy as np

# Dataset setup (Korr's Three Pillars)
# Input matrix shape: (4 sensings, 3 binary features each)
# Features per column: [foot shift, guard drop, exhale]
tells = np.array([
    [1, 0, 1],  # Sensing 0: foot shift, no guard drop, exhale
    [0, 1, 1],  # Sensing 1: no foot shift, guard drop, exhale
    [0, 0, 1],  # Sensing 2: only exhale
    [1, 1, 1]   # Sensing 3: all three (bluff)
])

# Target column vector: 1 = strike imminent, 0 = hold
# Formatted as a column matrix with shape (4, 1) for dot product math
strike = np.array([[1, 1, 0, 0]]).T


# Activation functions
def relu(x):
    # Elementwise Rectified Linear Unit (ReLU): converts negative values to 0
    return np.maximum(0, x)


def relu2deriv(y):
    # Derivative of ReLU: returns 1.0 for positive outputs y > 0, otherwise 0.0
    # Used during backprop to block gradients through inactive/dead neurons
    return (y > 0).astype(float)


# Training loop function using Stochastic Gradient Descent (SGD)
def train(tells, strike, alpha, epochs, hidden_size, seed):
    # Seed the NumPy random number generator for reproducible weight initialization
    np.random.seed(seed)
    
    # Randomly initialize weight matrices with values scaled in the range [-1.0, 1.0]
    # weights_0_1 connects inputs (3) to hidden neurons (hidden_size)
    weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
    # weights_1_2 connects hidden neurons (hidden_size) to output neuron (1)
    weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

    # Track total squared error at the end of each epoch
    error_history = []

    # Outer loop: iterate over total number of training epochs
    for epoch in range(epochs):
        total_error = 0.0
        
        # Inner loop: Stochastic Gradient Descent (SGD) — process sample by sample
        for i in range(len(tells)):
            # Extract single sample row while preserving 2D shape (1, 3)
            layer_0 = tells[i:i+1]   # shape (1, 3)
            # Extract single target value while preserving 2D shape (1, 1)
            target = strike[i:i+1]   # shape (1, 1)

            # Forward Pass
            # Compute hidden layer activations using ReLU applied to weighted sum
            layer_1 = relu(np.dot(layer_0, weights_0_1))  # shape (1, hidden_size)
            # Compute final output prediction (linear sum with no activation)
            layer_2 = np.dot(layer_1, weights_1_2)         # shape (1, 1)

            # Error Calculation
            # Calculate squared error for current sample: (prediction - target)^2
            layer_2_error = np.sum((layer_2 - target) ** 2)  # scalar float
            # Accumulate error to compute total error for full epoch
            total_error += layer_2_error

            # Backpropagation
            # Derivative of loss with respect to prediction: (pred - target)
            layer_2_delta = layer_2 - target  # shape (1, 1)
            
            # Backpropagate error through transposed weights_1_2 to hidden layer,
            # then multiply by relu2deriv to mask out inactive hidden units
            layer_1_delta = np.dot(layer_2_delta, weights_1_2.T) * relu2deriv(layer_1)  # shape (1, hidden_size)

            # SGD Weight Updates
            # Update weights_1_2 using outer product of layer_1 activation and layer_2_delta
            weights_1_2 -= alpha * np.dot(layer_1.T, layer_2_delta)  # shape (hidden_size, 1)
            # Update weights_0_1 using outer product of layer_0 input and layer_1_delta
            weights_0_1 -= alpha * np.dot(layer_0.T, layer_1_delta)  # shape (3, hidden_size)

        # Append total error recorded across all 4 samples for this epoch
        error_history.append(total_error)

    return weights_0_1, weights_1_2, error_history


if __name__ == "__main__":
    # Main run setup and training
    # Hyperparameters required by prompt
    alpha = 0.2
    epochs = 60
    hidden_size = 4
    seed = 1

    # Train model and get final weights and error history list
    w01, w12, error_history = train(tells, strike, alpha, epochs, hidden_size, seed)

    # Print total error every 10 epochs to observe convergence from ~1 to < 10^-3
    print("--- TOTAL ERROR EVERY 10 EPOCHS ---")
    for ep in range(0, epochs, 10):
        print(f"Epoch {ep:2d}: {error_history[ep]:.6f}")
    # Print error at final epoch 59
    print(f"Epoch 59: {error_history[-1]:.6f}\n")

    # Final predictions vs goals
    print("--- PREDICTIONS VS GOALS ---")
    # Evaluate model predictions for each sensing sample using learned weights
    for i in range(len(tells)):
        l0 = tells[i:i+1]                             # Input sample, shape (1, 3)
        l1 = relu(np.dot(l0, w01))                    # Hidden layer forward activation
        pred = np.dot(l1, w12)[0, 0]                  # Final prediction scalar
        goal = strike[i, 0]                           # Target goal scalar
        print(f"Sensing {i}: Pred = {pred:.4f} | Goal = {goal}")

    # Learned weights printout
    print("\n--- LEARNED WEIGHTS ---")
    # Display rounded weight matrices
    print("weights_0_1.round(2):\n", w01.round(2))
    print("weights_1_2.round(2):\n", w12.round(2))

    '''
    PART 4: HIDDEN LAYER INTERPRETATION (seed=1, hidden_size=4)
    Unit 0: Dead unit
    Unit 1: Detects foot shift but not guard drop
    Unit 2: Dead unit
    Unit 3: Detects guard drop but not foot shift
    '''

    # Hidden-size sweep execution
    print("\n--- HIDDEN-SIZE SWEEP ---")
    # Test sizes {1, 2, 4, 8, 16} across seeds {1, 2, 3}
    for hs in [1, 2, 4, 8, 16]:
        # For each size, train across 3 seeds and extract final epoch total error (index [2][-1])
        errors = [f"{train(tells, strike, alpha, epochs, hs, s)[2][-1]:.6f}" for s in [1, 2, 3]]
        print(f"Size {hs}: Seed 1 = {errors[0]}, Seed 2 = {errors[1]}, Seed 3 = {errors[2]}")

    '''
    PART 4: SWEEP CONCLUSION
    Sizes 1 and 2 fail every time, size 4 is a coin flip, and sizes 8 and 16 succeed reliably.
    The smallest size that works on seed 1 is 4, but the smallest that works reliably is 8 
    because smaller networks often suffer from dead ReLU units depending on weight initialization.
    '''


### Part 5 - Brady

In [ ]:
import numpy as np

from part1_single_layer_fails import single_layer_train
from part2_forward_hidden import relu, forward
from part3_one_backprop_step import relu2deriv, one_step
from part4_full_training_loop import train

## ----- Test 1 --------
def test_relu():
  result = relu(np.array([-1, 0, 1, 2]))
  expected = np.array([0, 0, 1, 2])
  
  assert np.array_equal(result, expected)



## ----- Test 2 ------
def test_relu2deriv():
  result = relu2deriv(np.array([-1, 0, 1, 2]))
  expected = np.array([0, 0, 1, 1])
  
  assert np.array_equal(result, expected)

## ----- Test 3 ------
def test_single_layer_failure():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  strike = np.array([1, 1, 0, 0])
  
  error = single_layer_train(tells, strike, 0.1, 60, 1)
  
  assert error > 0.5

## ----- Test 4 ------
def test_forward_layer1_shape():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  
  np.random.seed(1)
  hidden_size = 4
  
  weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
  weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1
  
  layer_1, layer_2 = forward(
    tells[0:1],
    weights_0_1,
    weights_1_2
  )
  
  assert layer_1.shape == (1, hidden_size)

## ---- Test 5 -------
def test_forward_layer2_shape():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  np.random.seed(1)
  hidden_size = 4
  
  weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
  weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1
  
  layer_1, layer_2 = forward(
    tells[0:1],
    weights_0_1,
    weights_1_2
  )
  
  assert layer_2.shape == (1, 1)

## ---- Test 6 ------
def test_one_step_reduce_error():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  
  strike = np.array([
    [1],
    [1],
    [0],
    [0]
  ])
  
  np.random.seed(1)
  hidden_size = 4
  
  weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
  weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1
  
  layer_0 = tells[0:1]
  target = strike[0:1]
  
  layer_1_before = relu(layer_0.dot(weights_0_1))
  layer_2_before = layer_1_before.dot(weights_1_2)
  
  error_before = np.sum((layer_2_before - target) ** 2)
  
  update_w01, update_w12, _, _ = one_step(
    layer_0, target, weights_0_1, weights_1_2, 0.2
  )
  
  layer_1_after = relu(layer_0.dot(update_w01))
  layer_2_after = layer_1_after.dot(update_w12)
  
  error_after = np.sum((layer_2_after - target) ** 2)
  
  assert error_after < error_before

## ----- Test 7 ------
def test_full_training_convergence():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  
  strike = np.array([
    [1],
    [1],
    [0],
    [0]
  ])

  weights_0_1, weights_1_2, error_history = train(
    tells, strike, alpha=0.2, epochs=60, hidden_size=4, seed=1
  )

  assert error_history[-1] < 1e-2

  for i in range(len(tells)):
    layer_0 = tells[i:i+1]
    layer_1 = relu(layer_0.dot(weights_0_1))
    layer_2 = layer_1.dot(weights_1_2)
    
    prediction = layer_2[0,0]
    
    if strike[i, 0] == 1:
      assert prediction > 0.5
    else:
      assert prediction < 0.5

## ----- Test 8 -----
def test_determinism():
  tells = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
  ])
  
  strike = np.array([
    [1],
    [1],
    [0],
    [0]
  ])

  weights_0_1_a, weights_1_2_a, _ = train(
    tells, strike, alpha=0.2, epochs=60, hidden_size=4, seed=1
  )

  weights_0_1_b, weights_1_2_b, _ = train(
    tells, strike, alpha=0.2, epochs=60, hidden_size=4, seed=1
  )

  assert np.allclose(weights_0_1_a, weights_0_1_b, atol=1e-10)
  assert np.allclose(weights_1_2_a, weights_1_2_b, atol=1e-10)


if __name__ == '__main__':
    tests = [name for name in dir() if name.startswith('test_')]
    passed = []
    failed = []
    for test_name in sorted(tests):
        test_func = globals()[test_name]
        try:
            test_func()
            passed.append(test_name)
            print(f' PASS: {test_name}')
        except AssertionError as e:
            failed.append(test_name)
            print(f' FAIL: {test_name} -- {e}')
    for name in passed:
        print(f'PASS: {name}')
    for name in failed:
        print(f'FAIL: {name}')
  
